In [1]:
import os

# Create data folders
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/clean", exist_ok=True)

print("✅ Folders created at:")
print(os.path.abspath("../data/raw"))
print(os.path.abspath("../data/clean"))

✅ Folders created at:
C:\Users\nizza\Documents\my-projects\malaysia-socioeconomic-portfolio\data\raw
C:\Users\nizza\Documents\my-projects\malaysia-socioeconomic-portfolio\data\clean


In [2]:
import subprocess
subprocess.run(["pip", "install", "requests", "pandas", "openpyxl"])

CompletedProcess(args=['pip', 'install', 'requests', 'pandas', 'openpyxl'], returncode=0)

In [5]:
import requests
import pandas as pd
import os
import time

# Create folders if they don't exist yet
os.makedirs("../data/raw",   exist_ok=True)
os.makedirs("../data/clean", exist_ok=True)

print("✅ Folders ready:")
print("  ", os.path.abspath("../data/raw"))
print("  ", os.path.abspath("../data/clean"))
print()

# ── Helper function ──────────────────────────────────────────
BASE_URL = "https://api.data.gov.my/opendosm"

def fetch_dosm(dataset_id, limit=1000, extra_params=None):
    """
    Fetch one dataset from OpenDOSM API.
    Saves CSV to ../data/raw/  and  returns a DataFrame.
    """
    params = {"id": dataset_id, "limit": limit}
    if extra_params:
        params.update(extra_params)

    try:
        response = requests.get(BASE_URL, params=params, timeout=30)

        if response.status_code == 200:
            data = response.json()

            # Handle both {"data": [...]}  and  direct list responses
            if isinstance(data, dict) and "data" in data:
                df = pd.DataFrame(data["data"])
            elif isinstance(data, list):
                df = pd.DataFrame(data)
            else:
                print(f"⚠️  {dataset_id}: Unexpected response format")
                return None

            # Save to CSV
            filepath = f"../data/raw/{dataset_id}.csv"
            df.to_csv(filepath, index=False)
            print(f"  ✅  {dataset_id:<40} {len(df):>5} rows  →  saved")
            return df

        else:
            print(f"  ❌  {dataset_id:<40} HTTP {response.status_code}")
            return None

    except requests.exceptions.ConnectionError:
        print(f"  ❌  {dataset_id:<40} No internet connection")
        return None
    except requests.exceptions.Timeout:
        print(f"  ❌  {dataset_id:<40} Request timed out — try again")
        return None
    except Exception as e:
        print(f"  ❌  {dataset_id:<40} Error: {e}")
        return None

print("✅  fetch_dosm() function ready!")
print()
print("👉  Now run Cell 2 below.")

✅ Folders ready:
   C:\Users\nizza\Documents\my-projects\malaysia-socioeconomic-portfolio\data\raw
   C:\Users\nizza\Documents\my-projects\malaysia-socioeconomic-portfolio\data\clean

✅  fetch_dosm() function ready!

👉  Now run Cell 2 below.


In [8]:
# ============================================================
# CELL 2 — TEMA 1 CORE: The Ageing Nation Story
# ============================================================

print("=" * 55)
print("TEMA 1 — CORE DATASETS (The Ageing Nation Story)")
print("=" * 55)

pop_malaysia    = fetch_dosm("population_malaysia")
pop_state       = fetch_dosm("population_state")
pop_district    = fetch_dosm("population_district")
fertility       = fetch_dosm("fertility")
fertility_state = fetch_dosm("fertility_state")
births     = fetch_dosm("births_annual")
deaths     = fetch_dosm("deaths")
marriages  = fetch_dosm("marriages")
hh_profile = fetch_dosm("hh_profile")

print()
print("✅  Tema 1 CORE done! Run Cell 3 next.")

TEMA 1 — CORE DATASETS (The Ageing Nation Story)
  ✅  population_malaysia                       1000 rows  →  saved
  ✅  population_state                          1000 rows  →  saved
  ✅  population_district                       1000 rows  →  saved
  ✅  fertility                                  528 rows  →  saved
  ✅  fertility_state                           1000 rows  →  saved
  ✅  births_annual                               24 rows  →  saved
  ✅  deaths                                      24 rows  →  saved
  ✅  marriages                                   12 rows  →  saved
  ✅  hh_profile                                  19 rows  →  saved

✅  Tema 1 CORE done! Run Cell 3 next.


In [9]:
# ============================================================
# CELL 4 — TEMA 2 CORE: Income vs Inflation Story
# ============================================================

print("=" * 55)
print("TEMA 2 — CORE DATASETS (Income vs Inflation Story)")
print("=" * 55)

hh_income           = fetch_dosm("hh_income")
hh_income_state     = fetch_dosm("hh_income_state")
cpi_headline        = fetch_dosm("cpi_headline",           limit=1000)
cpi_inflation       = fetch_dosm("cpi_headline_inflation", limit=1000)
hh_inequality       = fetch_dosm("hh_inequality")
hh_inequality_state = fetch_dosm("hh_inequality_state")
hh_poverty          = fetch_dosm("hh_poverty")
hh_poverty_state    = fetch_dosm("hh_poverty_state")
hies_state    = fetch_dosm("hies_state")
cpi_state     = fetch_dosm("cpi_state",     limit=1000)
cpi_lowincome = fetch_dosm("cpi_lowincome", limit=1000)
cpi_annual    = fetch_dosm("cpi_annual")
print()
print("✅  Tema 2 CORE done! Run Cell 5 next.")

TEMA 2 — CORE DATASETS (Income vs Inflation Story)
  ✅  hh_income                                   21 rows  →  saved
  ✅  hh_income_state                            303 rows  →  saved
  ✅  cpi_headline                              1000 rows  →  saved
  ✅  cpi_headline_inflation                    1000 rows  →  saved
  ✅  hh_inequality                               20 rows  →  saved
  ✅  hh_inequality_state                        273 rows  →  saved
  ✅  hh_poverty                                  20 rows  →  saved
  ✅  hh_poverty_state                           294 rows  →  saved
  ✅  hies_state                                  16 rows  →  saved
  ✅  cpi_state                                 1000 rows  →  saved
  ✅  cpi_lowincome                             1000 rows  →  saved
  ✅  cpi_annual                                 545 rows  →  saved

✅  Tema 2 CORE done! Run Cell 5 next.


In [12]:
# ============================================================
# VERIFY: Full download summary
# ============================================================

print("=" * 65)
print("  DOWNLOAD SUMMARY — ALL 21 DATASETS")
print("=" * 65)

all_datasets = {
    # ── Tema 1 Core ──────────────────────────────────────────
    "population_malaysia"     : pop_malaysia,
    "population_state"        : pop_state,
    "population_district"     : pop_district,
    "fertility"               : fertility,
    "fertility_state"         : fertility_state,
    "births_annual"           : births,
    "deaths"                  : deaths,
    "marriages"               : marriages,
    "hh_profile"              : hh_profile,
    # ── Tema 2 Core ──────────────────────────────────────────
    "hh_income"               : hh_income,
    "hh_income_state"         : hh_income_state,
    "cpi_headline"            : cpi_headline,
    "cpi_headline_inflation"  : cpi_inflation,
    "hh_inequality"           : hh_inequality,
    "hh_inequality_state"     : hh_inequality_state,
    "hh_poverty"              : hh_poverty,
    "hh_poverty_state"        : hh_poverty_state,
    "hies_state"              : hies_state,
    "cpi_state"               : cpi_state,
    "cpi_lowincome"           : cpi_lowincome,
    "cpi_annual"              : cpi_annual,
}

success = 0
failed  = []

print(f"  {'DATASET':<40} {'ROWS':>6}  {'COLUMNS'}")
print("  " + "-" * 60)

for name, df in all_datasets.items():
    if df is not None and len(df) > 0:
        cols = list(df.columns)
        print(f"  ✅  {name:<38} {len(df):>5} rows  |  {cols}")
        success += 1
    else:
        print(f"  ❌  {name:<38} FAILED or EMPTY")
        failed.append(name)

print()# ============================================================
# QUICK PREVIEW: Eyeball data
# ============================================================

print("── population_malaysia (first 3 rows) ──")
display(pop_malaysia.head(3))

print("── hh_income (first 3 rows) ──")
display(hh_income.head(3))

print("── cpi_headline_inflation (first 3 rows) ──")
display(cpi_inflation.head(3))

print("── hh_inequality_state (first 3 rows) ──")
display(hh_inequality_state.head(3))
print("=" * 65)
print(f"  Downloaded: {success} / {len(all_datasets)} datasets")
if failed:
    print(f"  ⚠️  Failed:   {failed}")
    print(f"  → Download these manually from open.dosm.gov.my")
    print(f"    and save the CSV into:  ../data/raw/")
else:
    print(f"  🎉  ALL 21 datasets downloaded successfully!")
print("=" * 65)

  DOWNLOAD SUMMARY — ALL 21 DATASETS
  DATASET                                    ROWS  COLUMNS
  ------------------------------------------------------------
  ✅  population_malaysia                     1000 rows  |  ['age', 'sex', 'date', 'ethnicity', 'population']
  ✅  population_state                        1000 rows  |  ['age', 'sex', 'date', 'state', 'ethnicity', 'population']
  ✅  population_district                     1000 rows  |  ['age', 'sex', 'date', 'state', 'district', 'ethnicity', 'population']
  ✅  fertility                                528 rows  |  ['date', 'age_group', 'fertility_rate']
  ✅  fertility_state                         1000 rows  |  ['date', 'state', 'age_group', 'fertility_rate']
  ✅  births_annual                             24 rows  |  ['abs', 'date', 'rate']
  ✅  deaths                                    24 rows  |  ['abs', 'date', 'rate']
  ✅  marriages                                 12 rows  |  ['abs', 'sex', 'date', 'rate']
  ✅  hh_profile      

,age,sex,date,ethnicity,population
0,overall,both,1970-01-01,overall,10881.8
1,0-4,both,1970-01-01,overall,1702.4
2,5-9,both,1970-01-01,overall,1690.3


── hh_income (first 3 rows) ──


,date,income_mean,income_median
0,1970-01-01,264,166
1,1974-01-01,362,227
2,1976-01-01,505,308


── cpi_headline_inflation (first 3 rows) ──


,date,division,inflation_mom,inflation_yoy
0,2011-11-01,13,0.4,3.2
1,2011-12-01,13,0.0,3.1
2,2012-01-01,13,-0.1,2.7


── hh_inequality_state (first 3 rows) ──


,date,gini,state
0,1974-01-01,0.439,Johor
1,1976-01-01,0.469,Johor
2,1979-01-01,0.442,Johor


  Downloaded: 21 / 21 datasets
  🎉  ALL 21 datasets downloaded successfully!


In [13]:
print("── population_malaysia (first 3 rows) ──")
display(pop_malaysia.head(3))

print("── hh_income (first 3 rows) ──")
display(hh_income.head(3))

print("── cpi_headline_inflation (first 3 rows) ──")
display(cpi_inflation.head(3))

print("── hh_inequality_state (first 3 rows) ──")
display(hh_inequality_state.head(3))

── population_malaysia (first 3 rows) ──


,age,sex,date,ethnicity,population
0,overall,both,1970-01-01,overall,10881.8
1,0-4,both,1970-01-01,overall,1702.4
2,5-9,both,1970-01-01,overall,1690.3


── hh_income (first 3 rows) ──


,date,income_mean,income_median
0,1970-01-01,264,166
1,1974-01-01,362,227
2,1976-01-01,505,308


── cpi_headline_inflation (first 3 rows) ──


,date,division,inflation_mom,inflation_yoy
0,2011-11-01,13,0.4,3.2
1,2011-12-01,13,0.0,3.1
2,2012-01-01,13,-0.1,2.7


── hh_inequality_state (first 3 rows) ──


,date,gini,state
0,1974-01-01,0.439,Johor
1,1976-01-01,0.469,Johor
2,1979-01-01,0.442,Johor
